In [1]:
# Import necessary libraries
import json
import re
import logging
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm.notebook import tqdm
from collections import Counter, defaultdict
import pandas as pd
from datetime import datetime
from pathlib import Path
import os

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Check GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [6]:
# Configuration settings
CONFIG = {
    "model_id": "meta-llama/Meta-Llama-3-8B",
    "use_quantization": True,
    "device": device,
    "max_new_tokens": 500,
    "temperature": 0.1,
    "input_file": "/output_entities2.json",  # Update this path
    "output_dir": "lulc_extraction_output",
    "max_sentences": 5,  # Set to limit processing (e.g., 50)
}

# Create output directory
os.makedirs(CONFIG["output_dir"], exist_ok=True)

# Define valid LULC relations
VALID_RELATIONS = [
    "CHANGE_TO",
    "INCREASES_BY",
    "DECREASES_BY",
    "CAUSES",
    "LOCATED_IN",
    "OCCURS_DURING",
    "MEASURES",
    "AFFECTS",
    "FROM_TO",
    "ENABLES"
]

print("Configuration loaded successfully")

Configuration loaded successfully


In [7]:
def load_label_studio_data(file_path):
    """Load data from Label Studio JSON format - FIXED VERSION"""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        print(f"✅ Loaded {len(data)} items from {file_path}")
        
        # Process data to extract sentences and entities
        processed_data = []
        
        for item in data:
            # Extract sentence text and clean it
            sentence = item.get('data', {}).get('text', '')
            # Remove the "#text': '" prefix and trailing quote if present
            if sentence.startswith("#text': '"):
                sentence = sentence[9:]  # Remove "#text': '"
            if sentence.endswith("'"):
                sentence = sentence[:-1]  # Remove trailing quote
            
            entities = []
            
            # Extract entities from annotations (not predictions!)
            if 'annotations' in item and item['annotations']:
                for annotation in item['annotations']:
                    if 'result' in annotation:
                        for result in annotation['result']:
                            if result['type'] == 'labels' and 'value' in result:
                                entity = {
                                    'text': result['value']['text'],
                                    'label': result['value']['labels'][0],
                                    'start_char': result['value']['start'],
                                    'end_char': result['value']['end']
                                }
                                entities.append(entity)
            
            processed_data.append({
                'sentence': sentence,
                'entities': entities,
                'original_data': item
            })
        
        # Print statistics
        total_entities = sum(len(item['entities']) for item in processed_data)
        sentences_with_entities = sum(1 for item in processed_data if item['entities'])
        
        print(f"📊 Processing Statistics:")
        print(f"  - Total sentences: {len(processed_data)}")
        print(f"  - Sentences with entities: {sentences_with_entities}")
        print(f"  - Total entities extracted: {total_entities}")
        print(f"  - Average entities per sentence: {total_entities/len(processed_data):.2f}")
        
        return processed_data
        
    except Exception as e:
        logger.error(f"Error loading data: {e}")
        import traceback
        traceback.print_exc()
        return []

# Load your data with the fixed function
input_data = load_label_studio_data(CONFIG["input_file"])

# Verify the fix worked
if input_data:
    print("\n🔍 First item check:")
    first_item = input_data[0]
    print(f"Sentence: {first_item['sentence'][:100]}...")
    print(f"Entities found: {len(first_item['entities'])}")
    if first_item['entities']:
        for ent in first_item['entities']:
            print(f"  - {ent['text']} ({ent['label']})")

2025-07-10 09:41:56,685 - ERROR - Error loading data: [Errno 2] No such file or directory: 'output_entities2.json'
Traceback (most recent call last):
  File "/tmp/ipykernel_3313947/4004933177.py", line 4, in load_label_studio_data
    with open(file_path, 'r', encoding='utf-8') as f:
  File "/home/raham/venv/lib/python3.8/site-packages/IPython/core/interactiveshell.py", line 284, in _modified_open
    return io_open(file, *args, **kwargs)
FileNotFoundError: [Errno 2] No such file or directory: 'output_entities2.json'


In [8]:
import os

# Check current working directory
print("Current working directory:", os.getcwd())

# Check if file exists in current directory
file_path = 'output_entities2.json'
if os.path.exists(file_path):
    print(f"✅ File found: {file_path}")
else:
    print(f"❌ File not found: {file_path}")
    
# List files in current directory to see what's available
print("\nFiles in current directory:")
for file in os.listdir('.'):
    if file.endswith('.json'):
        print(f"  📄 {file}")

Current working directory: /home/raham/ARENA 2025/lulc_extraction_output
❌ File not found: output_entities2.json

Files in current directory:
  📄 lulc_joint_extraction_20250709_105606.json
  📄 statistics_20250703_095929.json
  📄 relations_extracted_20250702_173348.json
  📄 statistics_20250709_165543.json
  📄 statistics_20250703_163009.json
  📄 joint_extraction_20250709_101201.json
  📄 lulc_relations_20250703_162528.json
  📄 statistics_20250709_0950322.json
  📄 relations_extracted_20250709_164517.json
  📄 relations_extracted_20250709_094955.json
  📄 label_studio_with_relations_20250702_173746.json
  📄 statistics_20250709_095654.json
  📄 relations_extracted_20250709_095032.json
  📄 relations_extracted_20250709_161857.json
  📄 statistics_20250709_164014.json
  📄 statistics_20250709_164419.json
  📄 lulc_relations_20250703_161346.json
  📄 statistics_20250703_172507.json
  📄 statistics_20250709_105618.json
  📄 relations_extracted_20250703_171420.json
  📄 lulc_joint_extraction_20250709_094954

In [4]:
def load_mistral_model(model_id, use_quantization=True):
    """Load Mistral model and tokenizer"""
    print(f"🔄 Loading model: {model_id}")
    
    try:
        # Load tokenizer
        tokenizer = AutoTokenizer.from_pretrained(model_id)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        
        # Configure quantization
        quantization_config = None
        if use_quantization and device == "cuda":
            quantization_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True
            )
            print("🔧 Using 4-bit quantization")
        
        # Load model
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto" if device == "cuda" else None,
            quantization_config=quantization_config,
            torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        )
        
        print(f"✅ Model loaded successfully on {device}")
        return model, tokenizer
        
    except Exception as e:
        logger.error(f"Failed to load model: {e}")
        raise

# Load the model
model, tokenizer = load_mistral_model(CONFIG["model_id"], CONFIG["use_quantization"])

🔄 Loading model: mistralai/Mistral-7B-Instruct-v0.2
🔧 Using 4-bit quantization


2025-07-09 09:47:18,964 - INFO - We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

✅ Model loaded successfully on cuda


In [5]:
def build_lulc_extraction_prompt(sentence):
    """Build prompt for joint entity recognition and relation extraction"""
    
    prompt = f"""You are an expert in Land Use Land Cover (LULC) analysis. Perform joint entity recognition and relation extraction from the given sentence.

SENTENCE: "{sentence}"

**Entity Types (USE ONLY THESE):**
- CHANGE: Transformation words (increased, decreased, converted, expanded, reduced, grew, declined, lost, gained, loss)
- LOC: Location names (countries, cities, specific name of districts, specific name of provinces)
- LULC: Land Use/Land Cover types (forest, cropland, urban area, built-up area, grassland, wetland, water body, bare ground, agricultural land, residential area, degraded land, woody vegetation)
- DATE: Temporal references (years, months, periods, seasons, decades, 1970s, 1980s)
- PERCENT: Percentage values (25%, 10.5%, thirty percent)
- CARDINAL: Numeric values without % (1000, 2.5 million, three, 500 hectares)
- COORDINATES: Geographic coordinates (40.7°N, latitude 23.5)
- SURFACE_UNIT: Area measurements (100 hectares, 50 km², 1000 acres)
- PROCESS: Environmental processes (deforestation, urbanization, expansion, drought, flooding, desertification, degradation)
- QUANTITY: Other quantities (rate of change, annual loss, total area, large parts)

**ENTITY EXTRACTION RULES:**
1. Extract entities as concise as possible (e.g., "loss" not "observed loss of woody vegetation")
2. List each date separately (e.g., "1970s" and "1980s" as two entities)
3. "desertification" is a PROCESS (the process of becoming desert), not LULC
4. "woody vegetation" or "woody vegetation cover" is LULC
**CHANGE_TO**: Indicates a direct transformation from one LULC type to another.
- ✅ CORRECT: forest --CHANGE_TO-- cropland (trees cut, land converted to farming)
- ✅ CORRECT: agricultural land --CHANGE_TO-- urban area (farmland developed into city)
- ✅ CORRECT: grassland --CHANGE_TO-- built-up area (grass removed, buildings constructed)
- ❌ WRONG: built-up area --CHANGE_TO-- built-up area (same type, just quantity change)
- ❌ WRONG: forest --CHANGE_TO-- forest (same type, just area change)

**INCREASES_BY/DECREASES_BY**: For quantitative changes within same LULC type
- ✅ CORRECT: built-up area --INCREASES_BY-- 12.77% (more built-up area, not transformation)
- ✅ CORRECT: forest --DECREASES_BY-- 25% (less forest area, not transformation)

**Other Relations:**
- CAUSES: Process entity causes a change (deforestation --CAUSES-- forest loss)
- LOCATED_IN: Spatial relationships (forest --LOCATED_IN-- Brazil)
- OCCURS_DURING: Temporal relationships (change --OCCURS_DURING-- 2018)
- MEASURES: Quantitative relationships (12.77% --MEASURES-- increase)
- AFFECTS: Impact relationships (urbanization --AFFECTS-- forest)
- FROM_TO: Value changes (52.88% --FROM_TO-- 65.5%)
- ENABLES: One process enables another (deforestation --ENABLES-- urbanization)

**Relationship Types - BE VERY THOUGHTFUL:**
...
- OCCURS_DURING: Temporal relationships where a **change, process** takes place or is observed within a specific time period. (e.g., change --OCCURS_DURING-- 2018, urbanization --OCCURS_DURING-- decade)
❌ WRONG: simulation results --OCCURS_DURING-- study period (results don't 'occur' in time, they are 'from' or 'valid for' a period)

**CRITICAL THINKING RULES:**
1. **Ask yourself**: Is this ACTUALLY a transformation between different land types?
2. **Think about the process**: What physical change happened to the land?
3. **Consider causality**: What caused what? Don't create meaningless loops
4. **Be precise with measurements**: Percentages usually MEASURE changes, not cause them
5. **Temporal logic**: Changes happen DURING time periods, not TO time periods
6. **Spatial logic**: Things are LOCATED_IN places, places don't transform to places

INSTRUCTIONS:
1. Extract ONLY relations that are explicitly stated or directly implied in the sentence
2. Use ONLY the entities provided above
3. Each relation must include the entity label in the format: entity_text:ENTITY_LABEL
4. Each relation must follow the format: source_entity:SOURCE_LABEL --RELATIONSHIP-- target_entity:TARGET_LABEL
5. Include confidence level (HIGH/MEDIUM/LOW) for each relation
6. Do not create relations between entities of the same type using TRANSFORMS_TO

**OUTPUT FORMAT:**
ENTITIES:
- entity_text | ENTITY_TYPE

RELATIONS:
- entity:ENTITY_TYPE --RELATIONSHIP-- entity:ENTITY_TYPE | CONF: confidence_level

Now perform joint entity recognition and relation extraction:"""
    
    return prompt

In [6]:
def generate_relations(sentence, model, tokenizer):
    """Generate entities and relations using the model"""
    
    # Build prompt - now only needs sentence
    prompt = build_lulc_extraction_prompt(sentence)
    
    # Tokenize
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=3000,
        padding=True
    )
    
    # Move to device
    if device == "cuda":
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=CONFIG["max_new_tokens"],
            temperature=CONFIG["temperature"],
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1
        )
    
    # Decode response
    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )
    
    return response

# Test generation with first sentence
if input_data:
    test_response = generate_relations(
        input_data[0]['sentence'],
        model,
        tokenizer
    )
    print("Model response:")
    print(test_response)

/home/raham/venv/lib/python3.8/site-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


Model response:


ENTITIES:
- droughts | EVENT
- 1970s, 1980s | DATE
- Sahel | LOC
- large parts | QUANTITY
- degraded land | LULC
- e.g | PROCESS

RELATIONS:
- droughts:EVENT --CAUSES-- loss:CHANGE | CONF: HIGH
- loss:CHANGE --OCCURS_DURING-- 1970s:DATE, 1980s:DATE | CONF: HIGH
- loss:CHANGE --CAUSES-- degraded_land:LULC | CONF: HIGH
- Sahel:LOC --LOCATED_IN-- Africa:LOC | CONF: MEDIUM
- large_parts:QUANTITY --EQUALS-- degraded_land:LULC | CONF: HIGH
- e.g:PROCESS --IS_A-- degradation:PROCESS | CONF: MEDIUM
- degradation:PROCESS --ENABLES-- loss:CHANGE | CONF: MEDIUM

Note: The confidence level can be determined based on the context and clarity of the sentence.


In [7]:
def parse_relations_from_response(response, sentence, log_all_relations=True):
    """Parse entities and relations from model response for joint approach"""
    entities = []
    relations = []
    
    lines = response.strip().split('\n')
    in_entities_section = False
    in_relations_section = False
    
    # First, parse entities from the response
    entity_dict = {}  # Map entity text to its type
    
    for line in lines:
        line = line.strip()
        
        # Check sections
        if 'ENTITIES:' in line.upper():
            in_entities_section = True
            in_relations_section = False
            continue
        elif 'RELATIONS:' in line.upper():
            in_relations_section = True
            in_entities_section = False
            continue
            
        # Parse entities
        if in_entities_section and '|' in line:
            # Parse entity format: "- entity_text | ENTITY_TYPE"
            match = re.match(r'^-?\s*(.+?)\s*\|\s*([A-Z_]+)', line)
            if match:
                entity_text = match.group(1).strip()
                entity_type = match.group(2).strip()
                entity_dict[entity_text] = entity_type
                entities.append({
                    'text': entity_text,
                    'label': entity_type
                })
                if log_all_relations:
                    logging.info(f"Extracted Entity: {entity_text} | {entity_type}")
            
        # Parse relations
        if in_relations_section and '--' in line:
            # Parse format: "entity:TYPE --RELATIONSHIP-- entity:TYPE | CONF: level"
            patterns = [
                r'^-?\s*(.+?):([A-Z_]+)\s*--([A-Z_]+)-->\s*(.+?):([A-Z_]+)\s*\|\s*CONF:\s*(\w+)',
                r'^-?\s*(.+?):([A-Z_]+)\s*--([A-Z_]+)--\s*(.+?):([A-Z_]+)\s*\|\s*CONF:\s*(\w+)',
                r'^-?\s*(.+?):([A-Z_]+)\s*--([A-Z_]+)--\s*(.+?):([A-Z_]+)',
                # Fallback patterns without types
                r'^-?\s*(.+?)\s*--([A-Z_]+)-->\s*(.+?)\s*\|\s*CONF:\s*(\w+)',
                r'^-?\s*(.+?)\s*--([A-Z_]+)--\s*(.+?)\s*\|\s*CONF:\s*(\w+)',
            ]
            
            for pattern in patterns:
                match = re.match(pattern, line)
                if match:
                    if len(match.groups()) >= 6:  # Full format with types
                        source_text = match.group(1).strip()
                        source_type = match.group(2).strip()
                        relationship = match.group(3).strip()
                        target_text = match.group(4).strip()
                        target_type = match.group(5).strip()
                        confidence = match.group(6).strip().upper() if len(match.groups()) >= 6 else "MEDIUM"
                    else:  # Fallback format without types
                        source_text = match.group(1).strip()
                        relationship = match.group(2).strip()
                        target_text = match.group(3).strip()
                        confidence = match.group(4).strip().upper() if len(match.groups()) >= 4 else "MEDIUM"
                        # Try to get types from entity_dict
                        source_type = entity_dict.get(source_text, "UNKNOWN")
                        target_type = entity_dict.get(target_text, "UNKNOWN")
                    
                    # Create relation
                    relation = {
                        'source': source_text,
                        'source_type': source_type,
                        'relationship': relationship,
                        'target': target_text,
                        'target_type': target_type,
                        'confidence': confidence
                    }
                    
                    if log_all_relations:
                        logging.info(f"Generated Relation: {source_text}:{source_type} --{relationship}-- {target_text}:{target_type} | CONF: {confidence}")
                    
                    # Validate relationship type
                    if relationship in VALID_RELATIONS:
                        relations.append(relation)
                    else:
                        logging.warning(f"Invalid relationship type: {relationship}")
                    break
    
    return {
        'entities': entities,
        'relations': relations
    }

In [8]:
# Test generation with first sentence
if input_data:
    test_response = generate_relations(
        input_data[0]['sentence'],
        model,
        tokenizer
    )
    print("Model response:")
    print(test_response)
    
    # Parse the response
    parsed_result = parse_relations_from_response(test_response, input_data[0]['sentence'])
    print("\nParsed Entities:")
    for entity in parsed_result['entities']:
        print(f"  - {entity['text']} | {entity['label']}")
    print("\nParsed Relations:")
    for rel in parsed_result['relations']:
        print(f"  - {rel['source']}:{rel['source_type']} --{rel['relationship']}-- {rel['target']}:{rel['target_type']} | CONF: {rel['confidence']}")

2025-07-09 09:48:05,521 - INFO - Extracted Entity: droughts | EVENT
2025-07-09 09:48:05,524 - INFO - Extracted Entity: 1970s, 1980s | DATE
2025-07-09 09:48:05,525 - INFO - Extracted Entity: Sahel | LOC
2025-07-09 09:48:05,526 - INFO - Extracted Entity: large parts | QUANTITY
2025-07-09 09:48:05,527 - INFO - Extracted Entity: degraded land | LULC
2025-07-09 09:48:05,528 - INFO - Extracted Entity: e.g | PROCESS
2025-07-09 09:48:05,529 - INFO - Generated Relation: droughts:EVENT --CAUSES-- loss:CHANGE | CONF: HIGH
2025-07-09 09:48:05,530 - INFO - Generated Relation: loss:CHANGE --OCCURS_DURING-- 1970s:DATE, 1980s:DATE | CONF: HIGH
2025-07-09 09:48:05,531 - INFO - Generated Relation: loss:CHANGE --CAUSES-- degraded_land:LULC | CONF: HIGH
2025-07-09 09:48:05,532 - INFO - Generated Relation: Sahel:LOC --LOCATED_IN-- Africa:LOC | CONF: MEDIUM
2025-07-09 09:48:05,533 - INFO - Generated Relation: large_parts:QUANTITY --EQUALS-- degraded_land:LULC | CONF: HIGH
2025-07-09 09:48:05,533 - WARNING -

Model response:


ENTITIES:
- droughts | EVENT
- 1970s, 1980s | DATE
- Sahel | LOC
- large parts | QUANTITY
- degraded land | LULC
- e.g | PROCESS

RELATIONS:
- droughts:EVENT --CAUSES-- loss:CHANGE | CONF: HIGH
- loss:CHANGE --OCCURS_DURING-- 1970s:DATE, 1980s:DATE | CONF: HIGH
- loss:CHANGE --CAUSES-- degraded_land:LULC | CONF: HIGH
- Sahel:LOC --LOCATED_IN-- Africa:LOC | CONF: MEDIUM
- large_parts:QUANTITY --EQUALS-- degraded_land:LULC | CONF: HIGH
- e.g:PROCESS --IS_A-- degradation:PROCESS | CONF: MEDIUM
- degradation:PROCESS --ENABLES-- loss:CHANGE | CONF: MEDIUM

Note: The confidence level can be determined based on the context and clarity of the sentence.

Parsed Entities:
  - droughts | EVENT
  - 1970s, 1980s | DATE
  - Sahel | LOC
  - large parts | QUANTITY
  - degraded land | LULC
  - e.g | PROCESS

Parsed Relations:
  - droughts:EVENT --CAUSES-- loss:CHANGE | CONF: HIGH
  - loss:CHANGE --OCCURS_DURING-- 1970s:DATE, 1980s:DATE | CONF: HIGH
  - loss:CHANGE --CAUSES-- degraded_

In [9]:
def process_all_sentences(input_data, model, tokenizer, max_sentences=None):
    """Process all sentences to extract LULC entities and relations"""
    
    # Limit processing if specified
    if max_sentences:
        input_data = input_data[:max_sentences]
        print(f"🔄 Processing {max_sentences} sentences (limited for testing)")
    else:
        print(f"🔄 Processing all {len(input_data)} sentences")
    
    all_results = []
    successful_extractions = 0
    failed_extractions = 0
    
    # Process each sentence
    for idx, item in enumerate(tqdm(input_data, desc="Extracting entities and relations")):
        sentence = item['sentence']
        
        try:
            # Generate entities and relations
            response = generate_relations(sentence, model, tokenizer)
            
            # Parse entities and relations from response
            parsed_result = parse_relations_from_response(response, sentence)
            
            # Store results
            result = {
                'sentence_id': idx,
                'sentence': sentence,
                'original_entities': item.get('entities', []),  # Keep original if available
                'extracted_entities': parsed_result['entities'],
                'model_response': response,
                'extracted_relations': parsed_result['relations'],
                'num_entities': len(parsed_result['entities']),
                'num_relations': len(parsed_result['relations']),
                'processing_timestamp': datetime.now().isoformat()
            }
            
            all_results.append(result)
            
            if parsed_result['relations']:
                successful_extractions += 1
                logger.info(f"✅ Sentence {idx}: Found {len(parsed_result['entities'])} entities and {len(parsed_result['relations'])} relations")
            else:
                failed_extractions += 1
                logger.warning(f"⚠️ Sentence {idx}: No valid relations extracted")
                
        except Exception as e:
            logger.error(f"❌ Error processing sentence {idx}: {e}")
            failed_extractions += 1
            
            # Store error result
            error_result = {
                'sentence_id': idx,
                'sentence': sentence,
                'original_entities': item.get('entities', []),
                'extracted_entities': [],
                'model_response': f"ERROR: {str(e)}",
                'extracted_relations': [],
                'num_entities': 0,
                'num_relations': 0,
                'processing_timestamp': datetime.now().isoformat(),
                'error': str(e)
            }
            all_results.append(error_result)
    
    # Print summary statistics
    print(f"\n📊 Processing Complete!")
    print(f"  - Total sentences processed: {len(input_data)}")
    print(f"  - Successful extractions: {successful_extractions}")
    print(f"  - Failed extractions: {failed_extractions}")
    print(f"  - Success rate: {(successful_extractions/(successful_extractions+failed_extractions)*100):.1f}%")
    
    # Calculate entity and relation statistics
    total_entities = sum(len(result['extracted_entities']) for result in all_results)
    total_relations = sum(len(result['extracted_relations']) for result in all_results)
    sentences_with_relations = sum(1 for result in all_results if result['extracted_relations'])
    
    print(f"  - Total entities extracted: {total_entities}")
    print(f"  - Total relations extracted: {total_relations}")
    print(f"  - Sentences with relations: {sentences_with_relations}")
    if sentences_with_relations > 0:
        print(f"  - Average entities per sentence: {total_entities/len(all_results):.2f}")
        print(f"  - Average relations per successful sentence: {total_relations/sentences_with_relations:.2f}")
    
    return all_results

# Process all sentences (or limited number for testing)
print("🚀 Starting joint entity and relation extraction...")
results = process_all_sentences(
    input_data, 
    model, 
    tokenizer, 
    max_sentences=CONFIG["max_sentences"]  # Set to None to process all
)

# Save results to JSON file
output_file = Path(CONFIG["output_dir"]) / f"lulc_joint_extraction_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"💾 Results saved to: {output_file}")

# Display sample results
print(f"\n🔍 Sample Results (first 3 successful extractions):")
successful_results = [r for r in results if r['extracted_relations']]
for i, result in enumerate(successful_results[:3]):
    print(f"\n--- Sample {i+1} ---")
    print(f"Sentence: {result['sentence'][:100]}...")
    print(f"Entities found: {result['num_entities']}")
    for ent in result['extracted_entities'][:5]:  # Show first 5 entities
        print(f"  • {ent['text']} | {ent['label']}")
    if result['num_entities'] > 5:
        print(f"  ... and {result['num_entities'] - 5} more entities")
    print(f"Relations found: {result['num_relations']}")
    for rel in result['extracted_relations']:
        print(f"  ➤ {rel['source']}:{rel['source_type']} --{rel['relationship']}--> {rel['target']}:{rel['target_type']} | {rel['confidence']}")

🚀 Starting joint entity and relation extraction...
🔄 Processing 5 sentences (limited for testing)


Extracting entities and relations:   0%|          | 0/5 [00:00<?, ?it/s]

2025-07-09 09:48:24,243 - INFO - Extracted Entity: droughts | EVENT
2025-07-09 09:48:24,246 - INFO - Extracted Entity: 1970s, 1980s | DATE
2025-07-09 09:48:24,248 - INFO - Extracted Entity: Sahel | LOC
2025-07-09 09:48:24,248 - INFO - Extracted Entity: large parts | QUANTITY
2025-07-09 09:48:24,249 - INFO - Extracted Entity: degraded land | LULC
2025-07-09 09:48:24,250 - INFO - Extracted Entity: e.g | PROCESS
2025-07-09 09:48:24,251 - INFO - Generated Relation: droughts:EVENT --CAUSES-- loss:CHANGE | CONF: HIGH
2025-07-09 09:48:24,252 - INFO - Generated Relation: loss:CHANGE --OCCURS_DURING-- 1970s:DATE, 1980s:DATE | CONF: HIGH
2025-07-09 09:48:24,253 - INFO - Generated Relation: loss:CHANGE --CAUSES-- degraded_land:LULC | CONF: HIGH
2025-07-09 09:48:24,253 - INFO - Generated Relation: Sahel:LOC --LOCATED_IN-- Africa:LOC | CONF: MEDIUM
2025-07-09 09:48:24,254 - INFO - Generated Relation: large_parts:QUANTITY --EQUALS-- degraded_land:LULC | CONF: HIGH
2025-07-09 09:48:24,255 - WARNING -


📊 Processing Complete!
  - Total sentences processed: 5
  - Successful extractions: 3
  - Failed extractions: 2
  - Success rate: 60.0%
  - Total entities extracted: 20
  - Total relations extracted: 13
  - Sentences with relations: 3
  - Average entities per sentence: 4.00
  - Average relations per successful sentence: 4.33
💾 Results saved to: lulc_extraction_output/lulc_joint_extraction_20250709_094954.json

🔍 Sample Results (first 3 successful extractions):

--- Sample 1 ---
Sentence: After the droughts in the 1970s and 1980s, the observed loss of woody vegetation cover was often con...
Entities found: 6
  • droughts | EVENT
  • 1970s, 1980s | DATE
  • Sahel | LOC
  • large parts | QUANTITY
  • degraded land | LULC
  ... and 1 more entities
Relations found: 5
  ➤ droughts:EVENT --CAUSES--> loss:CHANGE | HIGH
  ➤ loss:CHANGE --OCCURS_DURING--> 1970s:DATE, 1980s:DATE | HIGH
  ➤ loss:CHANGE --CAUSES--> degraded_land:LULC | HIGH
  ➤ Sahel:LOC --LOCATED_IN--> Africa:LOC | MEDIUM
  ➤ deg

In [10]:
def save_extraction_results(results, output_dir):
    """Save results in multiple formats"""
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Compute basic statistics
    stats = {
        'total_sentences_processed': len(results),
        'sentences_with_relations': sum(1 for r in results if r.get('extracted_relations')),
        'total_relations_extracted': sum(len(r.get('extracted_relations', [])) for r in results),
        'relation_types': {}
    }
    
    # Count relation types
    for result in results:
        for relation in result.get('extracted_relations', []):
            relation_type = relation.get('relationship', 'UNKNOWN')
            stats['relation_types'][relation_type] = stats['relation_types'].get(relation_type, 0) + 1
    
    # 1. Save detailed results
    detailed_output = {
        'metadata': {
            'extraction_date': datetime.now().isoformat(),
            'model': CONFIG['model_id'],
            'statistics': stats
        },
        'results': results
    }
    
    detailed_path = Path(output_dir) / f"relations_extracted_{timestamp}.json"
    with open(detailed_path, 'w', encoding='utf-8') as f:
        json.dump(detailed_output, f, indent=2, ensure_ascii=False)
    
    print(f"✅ Detailed results saved to: {detailed_path}")
    
    # 2. Save simplified CSV format
    csv_data = []
    for result in results:
        sentence_id = result.get('sentence_id', 'N/A')
        sentence = result.get('sentence', 'N/A')
        for relation in result.get('extracted_relations', []):
            csv_data.append({
                'sentence_id': sentence_id,
                'sentence': sentence,
                'source': relation['source'],
                'relationship': relation['relationship'],
                'target': relation['target'],
                'confidence': relation.get('confidence', 'MEDIUM')
            })
    
    if csv_data:
        df = pd.DataFrame(csv_data)
        csv_path = Path(output_dir) / f"relations_table_{timestamp}.csv"
        df.to_csv(csv_path, index=False)
        print(f"✅ CSV table saved to: {csv_path}")
    
    # 3. Save statistics
    stats_path = Path(output_dir) / f"statistics_{timestamp}2.json"
    with open(stats_path, 'w', encoding='utf-8') as f:
        json.dump(stats, f, indent=2)
    
    print(f"✅ Statistics saved to: {stats_path}")
    
    return detailed_path, csv_path if csv_data else None, stats_path

# Save all results
paths = save_extraction_results(results, CONFIG["output_dir"])

✅ Detailed results saved to: lulc_extraction_output/relations_extracted_20250709_094955.json
✅ CSV table saved to: lulc_extraction_output/relations_table_20250709_094955.csv
✅ Statistics saved to: lulc_extraction_output/statistics_20250709_0949552.json


In [11]:
def create_label_studio_output(results, original_data, output_dir):
    """Create Label Studio compatible output with relations - FIXED VERSION"""
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    label_studio_tasks = []
    
    for idx, (result, orig_item) in enumerate(zip(results, original_data)):
        # Start with predictions structure
        predictions = []
        entity_id_map = {}
        entity_counter = 0
        
        # Get the original text
        text = orig_item['original_data']['data']['text']
        
        # Step 1: Add entities from original annotations
        if 'annotations' in orig_item['original_data'] and orig_item['original_data']['annotations']:
            for annotation in orig_item['original_data']['annotations']:
                if 'result' in annotation:
                    for res in annotation['result']:
                        if res['type'] == 'labels':
                            entity_id = f"ent_{idx}_{entity_counter}"
                            entity_counter += 1
                            entity_id_map[res['value']['text']] = entity_id
                            
                            # Add entity prediction
                            entity_pred = res.copy()
                            entity_pred['id'] = entity_id
                            predictions.append(entity_pred)
        
        # Step 2: Check for entities from model extraction that might not be in annotations
        for entity in result.get('entities', []):
            if entity['text'] not in entity_id_map:
                entity_id = f"ent_{idx}_{entity_counter}"
                entity_counter += 1
                entity_id_map[entity['text']] = entity_id
                
                # Create new entity prediction
                entity_pred = {
                    'id': entity_id,
                    'type': 'labels',
                    'value': {
                        'start': entity['start_char'] if entity['start_char'] >= 0 else 0,
                        'end': entity['end_char'] if entity['end_char'] >= 0 else len(entity['text']),
                        'text': entity['text'],
                        'labels': [entity['label']]
                    },
                    'from_name': 'label',
                    'to_name': 'text'
                }
                predictions.append(entity_pred)
        
        # Step 3: Process relations and create missing entities
        for rel_idx, relation in enumerate(result['relations']):
            source = relation['source']
            target = relation['target']
            
            # If source entity not found, create it
            if source not in entity_id_map:
                entity_id = f"ent_{idx}_{entity_counter}"
                entity_counter += 1
                entity_id_map[source] = entity_id
                
                # Try to find position in text
                source_start = text.lower().find(source.lower())
                source_end = source_start + len(source) if source_start != -1 else -1
                
                # Create entity prediction
                entity_pred = {
                    'id': entity_id,
                    'type': 'labels',
                    'value': {
                        'start': source_start if source_start >= 0 else 0,
                        'end': source_end if source_end >= 0 else len(source),
                        'text': source,
                        'labels': ['ENTITY']  # Default label for missing entities
                    },
                    'from_name': 'label',
                    'to_name': 'text'
                }
                predictions.append(entity_pred)
                logger.info(f"Created new entity for relation source: {source}")
            
            # If target entity not found, create it
            if target not in entity_id_map:
                entity_id = f"ent_{idx}_{entity_counter}"
                entity_counter += 1
                entity_id_map[target] = entity_id
                
                # Try to find position in text
                target_start = text.lower().find(target.lower())
                target_end = target_start + len(target) if target_start != -1 else -1
                
                # Create entity prediction
                entity_pred = {
                    'id': entity_id,
                    'type': 'labels',
                    'value': {
                        'start': target_start if target_start >= 0 else 0,
                        'end': target_end if target_end >= 0 else len(target),
                        'text': target,
                        'labels': ['ENTITY']  # Default label for missing entities
                    },
                    'from_name': 'label',
                    'to_name': 'text'
                }
                predictions.append(entity_pred)
                logger.info(f"Created new entity for relation target: {target}")
            
            # Now add the relation
            relation_pred = {
                'from_id': entity_id_map[source],
                'to_id': entity_id_map[target],
                'type': 'relation',
                'labels': [relation['relationship']],
                'direction': 'right',
                'meta': {
                    'confidence': relation['confidence']
                }
            }
            predictions.append(relation_pred)
        
        # Create Label Studio task
        task = {
            'data': orig_item['original_data']['data'],
            'predictions': [{
                'model_version': f'lulc_relations_{timestamp}',
                'result': predictions
            }]
        }
        
        # Keep original annotations if present
        if 'annotations' in orig_item['original_data']:
            task['annotations'] = orig_item['original_data']['annotations']
        
        label_studio_tasks.append(task)
    
    # Save Label Studio format
    ls_path = Path(output_dir) / f"label_studio_with_relations_{timestamp}.json"
    with open(ls_path, 'w', encoding='utf-8') as f:
        json.dump(label_studio_tasks, f, indent=2, ensure_ascii=False)
    
    print(f"✅ Label Studio format saved to: {ls_path}")
    print(f"📊 Created {len(label_studio_tasks)} tasks with entities and relations")
    
    # Count statistics
    total_predictions = sum(len(task['predictions'][0]['result']) for task in label_studio_tasks)
    print(f"📊 Total predictions (entities + relations): {total_predictions}")
    
    # Create Label Studio config XML with all entity types
    config_xml = """<View>
  <Text name="text" value="$text"/>
  <Labels name="label" toName="text">
    <Label value="LULC" background="#FF6B6B"/>
    <Label value="DATE" background="#4ECDC4"/>
    <Label value="LOCATION" background="#45B7D1"/>
    <Label value="PERCENT" background="#96CEB4"/>
    <Label value="PROCESS" background="#FECA57"/>
    <Label value="QUANTITY" background="#FF9FF3"/>
    <Label value="CHANGE" background="#A55EEA"/>
    <Label value="ENTITY" background="#B4B4B4"/>
  </Labels>
  <Relations name="relation" toName="label">
    <Relation value="TRANSFORMS_TO" background="#FF6B6B"/>
    <Relation value="INCREASES_BY" background="#4ECDC4"/>
    <Relation value="DECREASES_BY" background="#45B7D1"/>
    <Relation value="CAUSES" background="#96CEB4"/>
    <Relation value="LOCATED_IN" background="#FECA57"/>
    <Relation value="OCCURS_DURING" background="#FF9FF3"/>
    <Relation value="MEASURES" background="#A55EEA"/>
    <Relation value="AFFECTS" background="#54A0FF"/>
    <Relation value="FROM_TO" background="#5F27CD"/>
    <Relation value="ENABLES" background="#00D2D3"/>
  </Relations>
</View>"""
    
    config_path = Path(output_dir) / "label_studio_config.xml"
    with open(config_path, 'w') as f:
        f.write(config_xml)
    
    print(f"✅ Label Studio config saved to: {config_path}")
    
    return ls_path

# Create Label Studio output with fixed function
ls_output_path = create_label_studio_output(results, input_data, CONFIG["output_dir"])

KeyError: 'relations'

In [ ]:
def create_minimal_test_file(output_dir):
    """Create a minimal test file to verify Label Studio import works"""
    
    # Minimal working example
    test_task = {
        "data": {
            "text": "The forest area decreased by 25% in Brazil during 2018."
        },
        "annotations": [{
            "id": "test_annotation_1",
            "completed_by": 1,
            "result": [
                {
                    "id": "entity_1",
                    "type": "labels",
                    "value": {
                        "start": 4,
                        "end": 15,
                        "text": "forest area",
                        "labels": ["LULC"]
                    },
                    "from_name": "label",
                    "to_name": "text"
                },
                {
                    "id": "entity_2",
                    "type": "labels",
                    "value": {
                        "start": 29,
                        "end": 32,
                        "text": "25%",
                        "labels": ["PERCENT"]
                    },
                    "from_name": "label",
                    "to_name": "text"
                },
                {
                    "id": "entity_3",
                    "type": "labels",
                    "value": {
                        "start": 36,
                        "end": 42,
                        "text": "Brazil",
                        "labels": ["LOCATION"]
                    },
                    "from_name": "label",
                    "to_name": "text"
                },
                {
                    "from_id": "entity_1",
                    "to_id": "entity_2",
                    "type": "relation",
                    "labels": ["DECREASES_BY"],
                    "direction": "right"
                }
            ]
        }]
    }
    
    # Save minimal test
    test_path = Path(output_dir) / "minimal_test.json"
    with open(test_path, 'w') as f:
        json.dump([test_task], f, indent=2)
    
    print(f"✅ Minimal test file saved to: {test_path}")
    print("Try importing this file first!")
    
    return test_path

# Create minimal test
minimal_test_path = create_minimal_test_file(CONFIG["output_dir"])